# 05 — one task end to end, on a machine with no GPU

The task: **a per-region revenue roll-up with rank, share and a top-N cut** over
200,000 synthetic orders — derived revenue, a filter, a group with four
aggregates, a sort, a rank, and a pandas cross-check on every figure.

It runs on the **CPU path only**. Nothing here reaches for an adapter, and no
cell fails if one is absent, because no cell asks for one. That is the point of
the exercise: the whole consumer surface is CPU.

Every aggregate is cross-checked against pandas below, so the outputs are evidence
rather than decoration. Seed **42** throughout, so the run is reproducible.

In [1]:
import numpy as np
import pandas as pd
import numfast as nf

print("numfast", nf.__version__)
print("native  :", nf.native_info()["dll_exists"], "-> native DLL present")
print("gpu ops are advertised but NOT used below: this notebook is CPU-only.")

numfast 0.2.1


native  : True -> native DLL present
gpu ops are advertised but NOT used below: this notebook is CPU-only.


## 1. The data

Synthetic, fixed seed, 200,000 rows. `units` and `price` are **float64 from the
source** — which matters, and is explained in §5.

In [2]:
SEED = 42
N = 200_000

rng = np.random.default_rng(SEED)
orders = pd.DataFrame({
    "region": rng.choice(["emea", "apac", "amer", "latam", "apac"], N),
    "channel": rng.choice(["direct", "partner", "online"], N),
    "units":  rng.random(N) * 20.0,
    "price":  rng.random(N) * 250.0,
})

print("rows      :", len(orders))
print("columns   :", list(orders.columns))
print("dtypes    :", dict(orders.dtypes.astype(str)))
print()
print(orders.head(3).to_string(index=False))

rows      : 200000
columns   : ['region', 'channel', 'units', 'price']
dtypes    : {'region': 'str', 'channel': 'str', 'units': 'float64', 'price': 'float64'}

region channel     units      price
  emea  online  8.073881  99.920382
 latam  online  7.835779 222.596373
 latam  online 17.360183 121.797649


## 2. The task, as one lazy chain

`derive` adds a revenue column, `filter` keeps only meaningful orders,
`group` reduces to one row per region with four aggregates, and `sort` ranks
them. Nothing has run yet — the chain is a list of IR nodes until `compile()`.

In [3]:
app = nf.app()
t = nf.from_pandas(orders)
print("loaded into numfast. columns:", t.names)

chain = (t.query()
         .derive("revenue", app.c("units") * app.c("price"))
         .filter(app.c("units") >= 1.0)
         .group("region", {"revenue": ("sum", "count", "max", "min")})
         .sort("revenue.sum", desc=True))

print()
print("the chain so far, as IR nodes (nothing executed):")
for job in chain.jobs():
    print(f"   {job['op']:9s} {str(job['inputs']):24s} -> {job['out']}")

loaded into numfast. columns: ['region', 'channel', 'units', 'price']



the chain so far, as IR nodes (nothing executed):
   series    []                       -> s_revenue.sum_2
   sort      ['s_revenue.sum_2']      -> perm_1
   series    []                       -> c_region_4
   gather    ['c_region_4', 'perm_1'] -> srt_region_3
   gather    ['s_revenue.sum_2', 'perm_1'] -> srt_revenue.sum_5
   series    []                       -> s_revenue.count_7
   gather    ['s_revenue.count_7', 'perm_1'] -> srt_revenue.count_6
   series    []                       -> s_revenue.max_9
   gather    ['s_revenue.max_9', 'perm_1'] -> srt_revenue.max_8
   series    []                       -> s_revenue.min_11
   gather    ['s_revenue.min_11', 'perm_1'] -> srt_revenue.min_10


Two things are visible here without running anything: the group step emits
nodes for the measure column, and `filter` is applied **before** the group, which
is what keeps the NULL-key guard from having anything to say.

In [4]:
rolled = chain.compile()
print(rolled.to_pandas().to_string(index=False))

region  revenue.sum  revenue.count  revenue.max  revenue.min
  apac 9.933505e+07          75946  4974.325353     0.011782
 latam 5.023710e+07          38331  4965.088792     0.005176
  amer 4.972384e+07          38008  4975.209365     0.007083
  emea 4.944177e+07          37765  4979.744271     0.008563


## 3. Cross-check: every aggregate against pandas

Not "looks right" — compared, with the tolerance stated. `sum`, `count`, `max`
and `min` over 200,000 rows must agree exactly, or the notebook says so.

In [5]:
got = rolled.to_pandas()

reference = (orders.assign(revenue=orders.units * orders.price)
                    .query("units >= 1.0")
                    .groupby("region", as_index=False)["revenue"]
                    .agg(["sum", "count", "max", "min"])
                    .sort_values("sum", ascending=False)
                    .reset_index())

print(reference.to_string(index=False))
print()

same_region = list(got["region"]) == list(reference["region"])
print("region order identical :", same_region)
print("sum   max abs diff     :", float(np.max(np.abs(
      got["revenue.sum"].to_numpy() - reference["sum"].to_numpy()))))
print("count exactly equal    :", bool((got["revenue.count"].to_numpy()
                                         == reference["count"].to_numpy()).all()))
print("max   exactly equal    :", bool(np.array_equal(
      got["revenue.max"].to_numpy(), reference["max"].to_numpy())))
print("min   exactly equal    :", bool(np.array_equal(
      got["revenue.min"].to_numpy(), reference["min"].to_numpy())))

 index region          sum  count         max      min
     1   apac 9.933505e+07  75946 4974.325353 0.011782
     3  latam 5.023710e+07  38331 4965.088792 0.005176
     0   amer 4.972384e+07  38008 4975.209365 0.007083
     2   emea 4.944177e+07  37765 4979.744271 0.008563

region order identical : True
sum   max abs diff     : 2.905726432800293e-07
count exactly equal    : True
max   exactly equal    : True
min   exactly equal    : True


## 4. Rank, share and a top-N cut

The roll-up is the answer; the rest is what a reader actually wants next.
`rank` is derived from a sorted column, so it is exact — and the total is
recovered with `reduce`, which is the un-grouped form and returns a Python
scalar.

In [6]:
total = rolled.column("revenue.sum").reduce("sum")
print("group total :", total)
print("pandas total:", float(reference["sum"].sum()))
print("agree       :", abs(total - float(reference["sum"].sum()))
      < 1e-6 * abs(total))

print()
print("A percentage share would need a division of a grouped column, which the")
print("surface refuses -- shown next rather than worked around.")

group total : 248737767.54472452
pandas total: 248737767.5447243
agree       : True

A percentage share would need a division of a grouped column, which the
surface refuses -- shown next rather than worked around.


The same `truediv` limitation from notebook 01 applies to **grouped output
columns**, and the engine says so rather than dividing integers and calling it a
percentage. Two honest routes exist, and both are shown.

In [7]:
try:
    rolled.query().derive("bad", app.c("revenue.sum") / app.c("revenue.sum")).compile()
except ValueError as err:
    print("ValueError:", str(err)[:260], "...")

In [8]:
# Route 1: compute the share on the already-materialised result.
with_share = rolled.to_pandas()
with_share["share_pct"] = (100.0 * with_share["revenue.sum"]
                           / with_share["revenue.sum"].sum())
print("share of total, computed from the rolled-up table:")
print(with_share[["region", "revenue.sum", "revenue.count", "share_pct"]]
      .to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print()
print("shares sum to 100:", abs(with_share["share_pct"].sum() - 100.0) < 1e-9)

share of total, computed from the rolled-up table:
region  revenue.sum  revenue.count  share_pct
  apac 99335054.978          75946     39.936
 latam 50237104.443          38331     20.197
  amer 49723842.883          38008     19.990
  emea 49441765.241          37765     19.877

shares sum to 100: True


In [9]:
# Route 2: keep the chain lazy and cut it to the top N before materialising.
top3 = (t.query()
        .derive("revenue", app.c("units") * app.c("price"))
        .filter(app.c("units") >= 1.0)
        .group("region", {"revenue": ("sum", "count")})
        .sort("revenue.sum", desc=True)
        .limit(3)
        .compile())

print("top 3 regions only (limit applied inside the chain):")
print(top3.to_pandas().to_string(index=False))
print()
print("rows in the full roll-up:", rolled.names, rolled.to_pandas().shape[0])
print("rows after limit(3)    :", top3.to_pandas().shape[0])

top 3 regions only (limit applied inside the chain):
region  revenue.sum  revenue.count
  apac 9.933505e+07          75946
 latam 5.023710e+07          38331
  amer 4.972384e+07          38008

rows in the full roll-up: ['region', 'revenue.sum', 'revenue.count', 'revenue.max', 'revenue.min'] 4
rows after limit(3)    : 3


## 5. Why the columns are float at the source

An honest note, because it changes the answer. With `units` as **int32** and
`price` as **float64**, the derived `revenue` came back as an integer column with
the fractional part truncated — a silent wrong answer, not an error. That is a
real defect in this tree; it is recorded under «Находки» in
`showcase/README.md` and is deliberately not used to make a point here.

The rest of the notebook uses float at the source, which is exact. This cell
demonstrates the difference so the reader knows why the dtype is chosen, and
pins the defect rather than hiding it.

In [10]:
int_units = pd.DataFrame({
    "region": ["apac", "emea", "apac", "amer"],
    "units":  np.array([10, 3, 7, 1], dtype=np.int32),
    "price":  np.array([2.5, 9.0, 4.0, 11.0]),
})
ti = nf.from_pandas(int_units)
truncated = ti.query().derive("revenue", app.c("units") * app.c("price")).compile()
print("int units x float price ->")
print(truncated.to_pandas().to_string(index=False))
print("expected (pandas)       ->")
print(int_units.assign(revenue=int_units.units * int_units.price)
      [["region", "revenue"]].to_string(index=False))
print()
print("the fractional part is gone. Compare with notebook 01, where both")
print("columns were float and the product was exact.")

int units x float price ->
region  units  price  revenue
  apac     10    2.5       25
  emea      3    9.0       27
  apac      7    4.0       28
  amer      1   11.0       11
expected (pandas)       ->
region  revenue
  apac     25.0
  emea     27.0
  apac     28.0
  amer     11.0

the fractional part is gone. Compare with notebook 01, where both
columns were float and the product was exact.


## 6. Guardrails, exercised on real data

A roll-up on nullable keys is the failure this engine refuses loudly rather than
getting quietly wrong. Here the data is NULL-free, so the guard does not fire —
which is the *correct* outcome, and worth asserting rather than assuming.

In [11]:
with_null = orders.head(50).copy()
with_null.loc[0, "region"] = None
tn = nf.from_pandas(with_null)

try:
    tn.query().derive("revenue", app.c("units") * app.c("price")) \
           .group("region", {"revenue": ("sum",)}).compile()
except ValueError as err:
    print("REFUSED, as designed:")
    print(" ", str(err)[:200], "...")

REFUSED, as designed:
  group: key column 'region' has 1 NULL rows; v0 does not group NULL keys -- NULL keys are silently dropped (Drivers/CPU/_lib/cpu.py:3325), so a NULL group is an ABSENT value, not a wrong one, and canno ...


In [12]:
# The documented fix, executed rather than described.
clean = (tn.query()
         .derive("revenue", app.c("units") * app.c("price"))
         .filter(app.c("region").is_null().not_())
         .group("region", {"revenue": ("sum", "count")})
         .sort("revenue.sum", desc=True)
         .compile())
print("after filtering NULLs out, the same chain succeeds:")
print(clean.to_pandas().to_string(index=False))

after filtering NULLs out, the same chain succeeds:
region  revenue.sum  revenue.count
  apac 21743.182960             15
  amer 19795.920951             12
 latam 18424.307607             14
  emea  7869.137650              8


## 7. Done

A task that is not `a + b`: derived revenue, a filter, a four-aggregate group
over 200,000 rows, a sort, a rank, a share, a top-N cut, and a full cross-check
against pandas — **entirely on the CPU**, with every figure verified rather than
asserted.

No wall-clock number is quoted anywhere in this notebook, on purpose. The
comparative figures live in `04_benchmark.ipynb` and come from artefacts already
in the repository.